# Gold Layer: Price Drop Alerts Fact Notebook
Monitors price changes across advertisement snapshots, computing days on market and percentage price drops.

## 1. Setup and Imports
Import standard library tools, SQLAlchemy, Polars, and application database models.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is available in system path
project_root = str(Path.cwd().parents[1])
if project_root not in sys.path:
    sys.path.append(project_root)

from sqlalchemy import select, insert
from sqlalchemy.orm import Session
import polars as pl

# Database configuration and model layer namespaces
from app.config import db_engine
from app.models import silver, gold

## 2. Query Silver Layer Historical Listings
Load advertisement snapshots with dates and prices from `silver.SilverCleanAd`.

In [ ]:
with db_engine.connect() as connection:
    df_silver_raw = pl.read_database(
        select(
            silver.SilverCleanAd.ad_id,
            silver.SilverCleanAd.title,
            silver.SilverCleanAd.url,
            silver.SilverCleanAd.price,
            silver.SilverCleanAd.date
        ),
        connection=connection
    )

## 3. Track Price Changes and Days on Market
Group listings by `ad_id` to compare initial vs current prices and calculate days active.

In [ ]:
df_price_drop_alerts = (
    df_silver_raw
    .sort(['ad_id', 'date'])
    .group_by('ad_id')
    .agg(
        pl.col('title').last().alias('title'),
        pl.col('url').last().alias('url'),
        pl.col('price').first().alias('initial_price'),
        pl.col('price').last().alias('current_price'),
        pl.col('date').first().alias('first_seen_date'),
        pl.col('date').last().alias('last_seen_date')
    )
    # Keep only listings that experienced price variations
    .filter(pl.col('initial_price') != pl.col('current_price'))
    # Calculate business indicators
    .with_columns(
        (
            pl.col('last_seen_date') - pl.col('first_seen_date')
        ).dt.total_days().cast(pl.Int32).alias('days_on_market'),
        (
            ((pl.col('current_price') - pl.col('initial_price')) / pl.col('initial_price')) * 100
        ).round(2).alias('price_change_pct')
    )
    .drop(['first_seen_date', 'last_seen_date'])
)

## 4. Persist to Gold Price Drop Alert Fact Table (`ft_gold_price_drop_alerts`)
Insert detected price drop alerts into `gold.FactPriceDropAlert`.

In [ ]:
if not df_price_drop_alerts.is_empty():
    with Session(db_engine) as session:
        session.execute(
            insert(gold.FactPriceChangeAlert), df_price_drop_alerts.to_dicts()
        )
        session.commit()
        print(f"Successfully recorded {len(df_price_drop_alerts)} price change alerts.")
else:
    print("No price drop alerts found.")